# 00 · Setup and tests

**Asset-level calibrated volatility bounds and conformal volatility management**

Run in order: `00 → 01 → 02 → 03 → 04 → 05 → 06`. Each notebook reads the files written by the previous one, so
you can stop and resume (for example after a Colab disconnect).

**Two runs, same notebooks.** First run everything with `project.mode: development` (data end at each panel's
`dev_end`; holdout dates never enter any file). After both authors sign off the design and the code is frozen, set
`mode: holdout`, `signoff: true` and `frozen_commit: <hash>` in `config/config.yaml` and run 01→06 once more. The
holdout run reuses γ and the CVM level chosen in development and stops if the configuration has changed.

This notebook mounts Drive, installs missing packages, prints the configuration and the pre-registration status,
and runs the automated tests in `tests/test_calvol.py`.

**Runtime:** GPU (T4) for notebook 02; CPU is fine for the others.

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

## Environment

In [ ]:
import torch, platform, arch, statsmodels, yfinance
print("Python", platform.python_version(), "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("arch", arch.__version__, "| statsmodels", statsmodels.__version__, "| yfinance", yfinance.__version__)

## Pre-registration status

In [ ]:
p = cfg["project"]
print("mode:", p["mode"], "| signoff:", p["signoff"], "| frozen_commit:", repr(p["frozen_commit"]))
print("design hash (must match between development and holdout):", pp.design_hash(cfg))
for name in ["assets", "ff_us", "ff_intl"]:
    c = cfg["panels"][name]
    print(f"{name:8s} start {c['start']}  dev_end {c['dev_end']}  end {c['end']}  -> data used now end at",
          c["dev_end"] if MODE == "development" else c["end"])

## Configuration in use

In [ ]:
import yaml
print(yaml.safe_dump({k: v for k, v in cfg.items() if not k.startswith("_")}, sort_keys=False))

## Kenneth French factor files

Notebook 01 downloads them from the Kenneth French data library. If the download is blocked, download these zip
files manually from https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html and place them in
`data/raw/french/` (names exactly as on the site):

In [ ]:
from calvol.data import FRENCH_FILES
for k, f in FRENCH_FILES.items():
    print(("present " if (P.french / f).exists() else "        ") + f)

## Automated tests (leakage, purge, conformal timing, VM look-ahead, pre-registration gate)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed - do not continue"